In [1]:
"""
need to pip install a few modules:
    pandas
    scikit-learn
    matplotlib

"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
# from xgboost import XGBClassifier

In [2]:
#load data
data_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

In [3]:
#view data
total_len = len(data_df)  # three possible classifications: GALAXY, STAR, QSO
# data_df.head(10)
# test_df.head(10)

In [4]:
# apply data transformations and feature engineering

# convert alpha and delta to radians, then use cos() and sin() to map from the 2D plane to a 3D sphere
alpha_rad = np.radians(data_df['alpha'])
delta_rad = np.radians(data_df['delta'])

data_df['X_loc'] = np.cos(delta_rad) * np.cos(alpha_rad)
data_df['Y_loc'] = np.cos(delta_rad) * np.sin(alpha_rad)
data_df['Z_loc'] = np.sin(delta_rad)

data_df = data_df.drop(columns=['alpha', 'delta'])

# adding in new features that are common subtractions of the different 'ugriz' filters

data_df['u_g_diff'] = data_df['u'] - data_df['g']
data_df['g_r_diff'] = data_df['g'] - data_df['r']
data_df['r_i_diff'] = data_df['r'] - data_df['i']
data_df['i_z_diff'] = data_df['i'] - data_df['z']


In [5]:
print(data_df.head())

   id          u          g          r          i          z  redshift  \
0   0  25.472123  21.895559  20.357926  19.257113  18.621057  0.408982   
1   1  20.778509  19.087062  17.587208  17.226067  16.786433  0.157976   
2   2  21.035203  21.079128  21.171840  20.582629  20.557366  2.823770   
3   3  23.305056  21.050736  19.017754  18.365658  17.914952  0.536099   
4   4  21.703158  19.471680  18.234449  17.899447  17.616185  0.555761   

  spectral_type galaxy_population   class     X_loc     Y_loc     Z_loc  \
0             M      Red_Sequence  GALAXY -0.808809  0.510631  0.291692   
1             M      Red_Sequence  GALAXY -0.519995  0.665835  0.535041   
2           O/B        Blue_Cloud     QSO -0.815680  0.002952  0.578496   
3             M      Red_Sequence  GALAXY -0.461171 -0.474536  0.749758   
4             M      Red_Sequence  GALAXY -0.741866  0.583034  0.331220   

   u_g_diff  g_r_diff  r_i_diff  i_z_diff  
0  3.576564  1.537632  1.100813  0.636056  
1  1.691447  1.4

In [6]:
#convert categorical variables to numeric dummy variables and do a test-train split
import model_helper_functions
# import importlib
# importlib.reload(model_helper_functions)
if(True):
    cleaned_df = model_helper_functions.cat_to_dummy(data_df,False)
    X_train, X_test, y_train, y_test = train_test_split(cleaned_df.drop(columns=['class']), cleaned_df['class'], test_size=0.25, random_state = 12345)
else: #no need to convert categorical to dummy encoding
    X_train, X_test, y_train, y_test = train_test_split(data_df.drop(columns=['class']), data_df['class'], test_size=0.25, random_state = 12345)

In [7]:
print("Length of training data split: ", len(X_train), "and", len(y_train))
print("Length of testing section: ", len(X_test), "and", len(y_test))

Length of training data split:  433010 and 433010
Length of testing section:  144337 and 144337


In [5]:
###
# XGboost code, copied from https://xgboost.readthedocs.io/en/stable/get_started.html
# ToDo: hyperparameter tuning, read up on what is actually happening behind the scenes
###

bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1)
# fit model
bst.fit(X_train, y_train)

##
# ToDo: visualize more
##

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
model_helper_functions.save_model_results(bst, X_test, y_test,"xgboost_validation")
model_helper_functions.save_model_results(bst, X_train, y_train,"xgboost_fit")

In [14]:
###
# Logistic regression code
###

first_model = LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=200)
first_model.fit(X_train, y_train)


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",200
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default

In [ ]:
# list out the model weights and intercepts for each class category:

weights = first_model.coef_
intercepts = first_model.intercept_

# put all the values into a dataframe with relevant column names
df_weights = pd.DataFrame(weights, columns=X_train.columns)
df_weights["intercept"] = intercepts
df_weights.index.name = "Class"

print(df_weights)

                 id         u         g         r         i         z  \
Class                                                                   
0     -4.405269e-07  0.575869  0.655925 -0.033523 -0.522489 -0.766046   
1      4.379974e-07 -1.023207 -0.343520  0.230546  0.560876  0.645534   
2      2.529379e-09  0.447336 -0.312406 -0.197024 -0.038388  0.120513   

       redshift     X_loc     Y_loc     Z_loc  ...  g_r_diff  r_i_diff  \
Class                                          ...                       
0     -0.329340 -0.157558 -0.021433 -0.037613  ...  0.689448  0.488967   
1      1.780588  0.059275 -0.007206  0.074953  ... -0.574066 -0.330331   
2     -1.451249  0.098284  0.028639 -0.037340  ... -0.115382 -0.158636   

       i_z_diff       A/F       G/K         M       O/B  Blue_Cloud  \
Class                                                                 
0      0.243557 -0.282791 -0.034311  0.429165 -0.124493   -0.208343   
1     -0.084656  0.161465 -0.078152 -0.310678  0.1

In [17]:
model_helper_functions.save_model_results(first_model, X_test, y_test,"logreg_validation")
model_helper_functions.save_model_results(first_model, X_train, y_train,"logreg_fit")

{'balanced_accuracy': 0.8729433879670992,
 'per_class': {0: {'precision': 0.9610913675551946,
   'recall': 0.8319829047753603,
   'f1': 0.8918889753448932,
   'support': 283120},
  1: {'precision': 0.8550361007008406,
   'recall': 0.9179092603536846,
   'f1': 0.8853578593117963,
   'support': 87988},
  2: {'precision': 0.5754988498368373,
   'recall': 0.8689379987722529,
   'f1': 0.6924121595963106,
   'support': 61902}},
 'macro_precision': 0.7972087726976241,
 'macro_recall': 0.8729433879670992,
 'macro_f1': 0.823219664751,
 'weighted_precision': 0.8844175289561262,
 'weighted_recall': 0.8547262187940232,
 'weighted_f1': 0.8620451526017953,
 'matthews_corrcoef': 0.7487838749717494,
 'cohen_kappa': 0.7381210856431502,
 'roc_auc_ovr': 0.9579323957704032,
 'confusion_matrix': array([[235551,  12701,  34868],
        [  2415,  80765,   4808],
        [  7121,    992,  53789]]),
 'y_pred': array([0, 1, 1, ..., 0, 0, 2], shape=(433010,))}

In [12]:
# Extract weights from all classes in the model
weights = np.array([estimator.coef_[0] for estimator in first_model.estimators_])

# Extract biases (intercepts) for all classes
biases = np.array([estimator.intercept_[0] for estimator in first_model.estimators_])

for target_class, w, b in zip(first_model.classes_, weights, biases):
    print(f"Class {target_class} -> Bias: {b:.4f} | Weights: {w}")

Class 0 -> Bias: 0.0002 | Weights: [ 7.99804608e-07  5.15928829e-03  4.05981495e-03  2.77064879e-03
  2.03760895e-03  1.67533799e-03 -5.52618560e-04 -2.02455326e-04
 -3.37116544e-05  1.82228778e-05  1.09947334e-03  1.28916617e-03
  7.33039840e-04  3.62270954e-04 -4.22256662e-04 -4.69087479e-05
  8.15210116e-04 -1.90909572e-04 -5.81371513e-04  7.36506648e-04]
Class 1 -> Bias: -0.0016 | Weights: [-1.24223596e-06 -5.41395810e-02 -2.39430662e-02  3.39303896e-03
  1.80355142e-02  2.39808444e-02  3.76031137e-02  3.71585198e-03
  5.41419545e-04  1.43866645e-03 -3.01965148e-02 -2.73361052e-02
 -1.46424752e-02 -5.94533015e-03  9.78441252e-03 -9.32909472e-04
 -1.65062522e-02  6.05644611e-03  1.46597686e-02 -1.62580717e-02]
Class 2 -> Bias: -0.0001 | Weights: [ 1.05826121e-07 -2.82475340e-02 -2.66732582e-02 -1.61674882e-02
 -9.65308762e-03 -5.45089791e-03 -1.27299195e-02  2.64239694e-03
  6.32055577e-04 -6.81937489e-04 -1.57427582e-03 -1.05057700e-02
 -6.51440054e-03 -4.20218971e-03  4.56984521e-

In [ ]:
# transform the test dataset like we did with the training data

# convert alpha and delta to radians, then use cos() and sin() to map from the 2D plane to a 3D sphere
alpha_rad = np.radians(test_df['alpha'])
delta_rad = np.radians(test_df['delta'])

test_df['X_loc'] = np.cos(delta_rad) * np.cos(alpha_rad)
test_df['Y_loc'] = np.cos(delta_rad) * np.sin(alpha_rad)
test_df['Z_loc'] = np.sin(delta_rad)

test_df = test_df.drop(columns=['alpha', 'delta'])

# adding in new features that are common subtractions of the different 'ugriz' filters

test_df['u_g_diff'] = test_df['u'] - test_df['g']
test_df['g_r_diff'] = test_df['g'] - test_df['r']
test_df['r_i_diff'] = test_df['r'] - test_df['i']
test_df['i_z_diff'] = test_df['i'] - test_df['z']

In [ ]:
# set up dummy variables to ensure everything is numerical for the model
spectral = pd.get_dummies(test_df['spectral_type'], dtype=int)
galaxy_pop = pd.get_dummies(test_df['galaxy_population'], dtype=int)
    
# drop original categorical columns and concatenate the dummy variables to feed into logistic regression
temp_df = test_df.drop(columns=['spectral_type', 'galaxy_population'])
fin_test_df = pd.concat([temp_df, spectral, galaxy_pop], axis=1)

In [24]:
print(fin_test_df.head())

       id          u          g          r          i          z  redshift  \
0  577347  23.668066  21.951680  21.086183  20.180032  19.202124  0.429042   
1  577348  24.902933  22.338822  20.732163  19.860330  19.687691  0.867305   
2  577349  19.427591  18.474633  17.551314  16.570674  16.176765  0.224234   
3  577350  23.121029  21.526855  20.670159  20.417633  20.699095  0.066507   
4  577351  25.094282  22.643981  21.123173  19.439500  19.094158  0.977218   

      X_loc     Y_loc     Z_loc  u_g_diff  g_r_diff  r_i_diff  i_z_diff  A/F  \
0 -0.466950  0.785815  0.405528  1.716387  0.865497  0.906151  0.977908    0   
1 -0.572583 -0.470566  0.671354  2.564112  1.606658  0.871833  0.172640    0   
2 -0.993240  0.111959 -0.030650  0.952958  0.923319  0.980640  0.393909    0   
3 -0.996037 -0.085460 -0.024625  1.594174  0.856696  0.252526 -0.281462    0   
4 -0.711008 -0.651240  0.265243  2.450301  1.520808  1.683673  0.345342    0   

   G/K  M  O/B  Blue_Cloud  Red_Sequence  
0    1 

In [ ]:
# predict the classes of the testing data
final_predicts = first_model.predict(fin_test_df)


In [ ]:
# put the results into a DataFrame with 'id', can save to a csv and submit if needed

logis_df = pd.DataFrame({'id': fin_test_df['id'], 'class': final_predicts})
logis_df = model_helper_functions.dummy_to_cat(logis_df)

In [ ]:
# verify the resulting df looks correct
print(logis_df.head(20))

        id   class
0   577347  GALAXY
1   577348  GALAXY
2   577349  GALAXY
3   577350    STAR
4   577351  GALAXY
5   577352  GALAXY
6   577353  GALAXY
7   577354    STAR
8   577355  GALAXY
9   577356  GALAXY
10  577357    STAR
11  577358  GALAXY
12  577359     QSO
13  577360  GALAXY
14  577361     QSO
15  577362     QSO
16  577363    STAR
17  577364    STAR
18  577365     QSO
19  577366  GALAXY
